# Lesson 3 — Time-Series Analysis

**LIGO–Virgo–KAGRA Python Course**

---

> *"The signal buried in the noise is not found by looking harder — it is found by understanding the noise itself."*  
> — Adapted from the gravitational-wave data-analysis community

---

## Table of Contents

1. [Introduction](#1-introduction)
2. [The GWpy `TimeSeries` Object](#2-the-gwpy-timeseries-object)  
   2.1 [Loading Strain Data](#21-loading-strain-data)  
   2.2 [Core Attributes and Metadata](#22-core-attributes-and-metadata)  
   2.3 [Array Operations and Arithmetic](#23-array-operations-and-arithmetic)  
3. [Cropping and Time-Slicing](#3-cropping-and-time-slicing)  
   3.1 [The `crop()` Method](#31-the-crop-method)  
   3.2 [Index-Based Slicing](#32-index-based-slicing)  
   3.3 [Padding and Extending](#33-padding-and-extending)  
4. [Resampling](#4-resampling)  
   4.1 [Why Resample?](#41-why-resample)  
   4.2 [Downsampling with `resample()`](#42-downsampling-with-resample)  
   4.3 [Aliasing and the Nyquist Theorem](#43-aliasing-and-the-nyquist-theorem)  
   4.4 [Anti-Aliasing Filters](#44-anti-aliasing-filters)  
5. [Spectral Analysis](#5-spectral-analysis)  
   5.1 [The Discrete Fourier Transform](#51-the-discrete-fourier-transform)  
   5.2 [Power Spectral Density and Welch's Method](#52-power-spectral-density-and-welchs-method)  
   5.3 [Amplitude Spectral Density](#53-amplitude-spectral-density)  
   5.4 [Spectral Lines and Noise Features](#54-spectral-lines-and-noise-features)  
6. [Band-Pass Filtering](#6-band-pass-filtering)  
   6.1 [Filter Design Concepts](#61-filter-design-concepts)  
   6.2 [Butterworth Band-Pass Filter](#62-butterworth-band-pass-filter)  
   6.3 [Edge Effects and Windowing](#63-edge-effects-and-windowing)  
   6.4 [Notch Filters for Spectral Lines](#64-notch-filters-for-spectral-lines)  
7. [Whitening](#7-whitening)  
   7.1 [The Noise Floor Problem](#71-the-noise-floor-problem)  
   7.2 [Theoretical Foundation](#72-theoretical-foundation)  
   7.3 [Whitening with GWpy](#73-whitening-with-gwpy)  
   7.4 [Whitening Parameters and Trade-offs](#74-whitening-parameters-and-trade-offs)  
8. [Visualisation — GW150914](#8-visualisation--gw150914)  
   8.1 [Raw vs Band-Passed Strain](#81-raw-vs-band-passed-strain)  
   8.2 [Whitened Strain](#82-whitened-strain)  
   8.3 [Q-Transform of the Processed Data](#83-q-transform-of-the-processed-data)  
9. [Visualisation — GW170817](#9-visualisation--gw170817)  
   9.1 [Characteristics of a Binary Neutron Star Signal](#91-characteristics-of-a-binary-neutron-star-signal)  
   9.2 [Raw vs Whitened Strain for GW170817](#92-raw-vs-whitened-strain-for-gw170817)  
   9.3 [Comparing GW150914 and GW170817 in the Time Domain](#93-comparing-gw150914-and-gw170817-in-the-time-domain)  
10. [Handling Gaps, Data Artifacts, and GPS Time Conventions](#10-handling-gaps-data-artifacts-and-gps-time-conventions)  
    10.1 [Understanding Data Gaps](#101-understanding-data-gaps)  
    10.2 [Gap Filling Strategies](#102-gap-filling-strategies)  
    10.3 [Common Data Artifacts](#103-common-data-artifacts)  
    10.4 [GPS Time Conventions in Practice](#104-gps-time-conventions-in-practice)  
    10.5 [Segment Lists and the `DataQualityFlag`](#105-segment-lists-and-the-dataqualityflag)  
11. [Advanced Topics](#11-advanced-topics)  
    11.1 [The Q-Transform Revisited](#111-the-q-transform-revisited)  
    11.2 [Stationarity and Non-Stationarity](#112-stationarity-and-non-stationarity)  
    11.3 [Online vs Offline Processing](#113-online-vs-offline-processing)  
12. [Student Exercises](#12-student-exercises)
13. [References](#13-references)

---
## 1. Introduction

In the previous two lessons we built the physical intuition for gravitational waves and learned how to access real detector data through GWOSC. We know that raw strain data looks like featureless noise — the GW signals are buried far below the detector noise floor. This lesson is devoted to the signal-processing toolkit that makes those signals visible.

The central object in our Python workflow is **GWpy's `TimeSeries`** — a NumPy array subclass that carries physical metadata (GPS start time, sample rate, units) and exposes a rich set of methods for manipulation, spectral analysis, and visualisation. We will work through every major operation in depth:

| Operation | Purpose |
|---|---|
| **Cropping** | Select a time window of interest from a longer stretch |
| **Resampling** | Change the sample rate to trade bandwidth for file size / speed |
| **Band-pass filtering** | Pass only the frequency band where the detector is sensitive and signals live |
| **Whitening** | Flatten the noise spectrum so all frequencies are weighted equally |
| **Visualisation** | Plot time-domain strain and time–frequency spectrograms |

### Two Flagship Events

Throughout this lesson we will use two real events as worked examples:

- **GW150914** — the first binary black-hole merger ever detected (14 September 2015). A short, loud, chirp lasting ≲ 0.2 s in-band.
- **GW170817** — the first binary neutron-star merger (17 August 2017). A long, sweeping chirp that enters the LIGO band at ~25 Hz and sweeps up to ~1500 Hz over ~100 s, accompanied by a gamma-ray burst.

These two events illustrate opposite extremes of the signal-processing challenges we face.

### Prerequisites

You should be comfortable with:
- Lessons 1 and 2 of this course
- Basic NumPy array operations
- The concept of a Fourier transform
- (Optional) Elementary filter theory

All code is runnable on **Google Colab** or a local Python ≥ 3.9 environment. We will install any missing packages in the first code cell.

---
## 2. The GWpy `TimeSeries` Object

### 2.1 Loading Strain Data

GWpy's `TimeSeries` is the fundamental container for detector strain data. It wraps a NumPy array and adds:

- A **GPS start time** (`t0`) — the absolute time of the first sample
- A **sample rate** (`sample_rate`) — samples per second, in Hz
- A **unit** — typically dimensionless strain, annotated as `''` or `dimensionless`
- A **name** — the channel name, e.g. `'H1:GWOSC-4KHZ_R1_STRAIN'`
- All standard NumPy array operations through inheritance

The easiest way to load public LIGO/Virgo data is `TimeSeries.fetch_open_data()`, which calls the GWOSC API and returns the strain as a `TimeSeries`.

In [ ]:
# Cell 1 — Install required packages (run once; harmless if already installed)
import subprocess, sys

packages = ["gwpy", "gwosc", "astropy", "scipy", "matplotlib"]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pkg], check=False)

In [ ]:
# Cell 2 — Standard imports
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from gwpy.timeseries import TimeSeries
from gwpy.time import to_gps, from_gps
import astropy.units as u

# Matplotlib style — consistent with the rest of this course
matplotlib.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

print("All imports OK.")

In [ ]:
# Cell 3 — Fetch 32 seconds of H1 strain around GW150914
# GPS 1126259462 is the merger time; we grab ±16 s
GW150914_GPS = 1126259462.4

h1_150914 = TimeSeries.fetch_open_data(
    "H1",
    GW150914_GPS - 16,
    GW150914_GPS + 16,
    sample_rate=4096,
    cache=True,
)
print(h1_150914)

### 2.2 Core Attributes and Metadata

Once loaded, `TimeSeries` exposes a rich set of attributes. Let us inspect them:

In [ ]:
# Cell 4 — Inspect TimeSeries metadata
ts = h1_150914

print(f"Channel name  : {ts.name}")
print(f"GPS t0        : {ts.t0}  ({float(ts.t0):.6f} s)")
print(f"GPS t_end     : {ts.times[-1]:.6f}")
print(f"Sample rate   : {ts.sample_rate}")
print(f"Duration      : {ts.duration}")
print(f"N samples     : {len(ts)}")
print(f"dtype         : {ts.dtype}")
print(f"unit          : {ts.unit}")
print(f"Peak amplitude: {float(np.max(np.abs(ts.value))):.3e}")
print()
print("First 5 time stamps (GPS):", ts.times[:5])
print("Shape of underlying array:", ts.value.shape)

Key points:

- `ts.t0` is a `Quantity` in GPS seconds. Wrap it in `float()` to get a plain number.
- `ts.times` returns an array of GPS times for every sample — very useful for axis labels.
- `ts.value` strips the units and returns a plain NumPy array — required for SciPy DSP functions.
- `ts.duration` is `N_samples / sample_rate`, expressed as a `Quantity` in seconds.

### 2.3 Array Operations and Arithmetic

In [ ]:
# Cell 5 — Arithmetic on a TimeSeries
# Multiply by 1e21 to express strain in units of 10^{-21}
ts_scaled = ts * 1e21
print(f"After scaling: mean = {float(np.mean(ts_scaled)):.4f}, "
      f"std = {float(np.std(ts_scaled)):.4f} (units: {ts_scaled.unit})")

# Compute RMS
rms = float(np.sqrt(np.mean(ts.value**2)))
print(f"RMS strain: {rms:.3e} (dimensionless)")

# TimeSeries supports standard NumPy ufuncs
ts_abs = np.abs(ts)
print(f"Max |strain|: {float(ts_abs.max()):.3e}")

---
## 3. Cropping and Time-Slicing

### 3.1 The `crop()` Method

The most natural way to extract a sub-segment from a `TimeSeries` is the **`crop()`** method. It accepts GPS start and end times and returns a new `TimeSeries` with updated metadata:

In [ ]:
# Cell 6 — Cropping with GPS times
# Zoom in to ±1 s around the merger
ts_zoom = ts.crop(
    GW150914_GPS - 1.0,
    GW150914_GPS + 1.0,
)

print(f"Original duration : {float(ts.duration):.1f} s ({len(ts)} samples)")
print(f"Cropped duration  : {float(ts_zoom.duration):.4f} s ({len(ts_zoom)} samples)")
print(f"New t0 (GPS)      : {float(ts_zoom.t0):.6f}")
print(f"New t_end (GPS)   : {float(ts_zoom.times[-1]):.6f}")

# Plot the zoomed window
fig, ax = plt.subplots(figsize=(11, 3))
times_rel = ts_zoom.times.value - GW150914_GPS  # seconds relative to merger
ax.plot(times_rel, ts_zoom.value * 1e21, lw=0.8, color="#4575b4")
ax.set_xlabel("Time relative to GW150914 merger [s]")
ax.set_ylabel(r"Strain [$\times 10^{-21}$]")
ax.set_title("GW150914 H1 — raw strain (zoomed to ±1 s)")
ax.axvline(0, color="crimson", lw=1.5, ls="--", label="merger")
ax.legend()
plt.tight_layout()
plt.show()

Notice that the GW signal is **not visible** in the raw strain — it is completely swamped by instrument noise. We will fix that in Sections 6 and 7.

### 3.2 Index-Based Slicing

GWpy also supports standard NumPy index slicing, which is useful when you know the sample number rather than the GPS time:

In [ ]:
# Cell 7 — Index-based slicing
fs = int(ts.sample_rate.value)  # 4096 samples/s

# Grab the first 4 seconds (= first 4*4096 samples)
ts_first4 = ts[:4 * fs]
print(f"Slice [0 : 4*fs] → {len(ts_first4)} samples, duration = {float(ts_first4.duration):.3f} s")

# Grab the last 4 seconds
ts_last4 = ts[-4 * fs:]
print(f"Slice [-4*fs :]  → {len(ts_last4)} samples, duration = {float(ts_last4.duration):.3f} s")

# Important: index slicing does NOT update t0 in older GWpy versions.
# crop() is the safer choice for GPS-referenced work.
print(f"\nt0 after index slice : {float(ts_first4.t0):.6f}")
print(f"t0 after crop()      : {float(ts.crop(float(ts.t0), float(ts.t0)+4).t0):.6f}")

> **Best practice:** Always use `crop()` with GPS times when the absolute time reference matters (e.g., when comparing two detectors). Index slicing is fine for quick inspection.

### 3.3 Padding and Extending

Sometimes you need to *extend* a segment — for example, to provide buffer for filter edge effects (we call these "guard segments"). GWpy provides `pad()`:

In [ ]:
# Cell 8 — Padding a TimeSeries
# Add 0.5 s of zeros on each side
ts_padded = ts_zoom.pad(int(0.5 * fs))
print(f"Zoom duration    : {float(ts_zoom.duration):.3f} s")
print(f"Padded duration  : {float(ts_padded.duration):.3f} s")
print(f"Padded N samples : {len(ts_padded)}")

# The padding value can also be set to a constant
ts_padded_const = ts_zoom.pad(int(0.5 * fs), mode='mean')
print(f"Padding mode 'mean' — edge value: {float(ts_padded_const[:5].mean()):.3e}")

---
## 4. Resampling

### 4.1 Why Resample?

GWOSC data is available at 4096 Hz and 16384 Hz. The native sample rate choices are:

| Rate | Nyquist limit | Typical use case |
|---|---|---|
| **4096 Hz** | 2048 Hz | Standard GW data analysis (BBH, BNS) |
| **16384 Hz** | 8192 Hz | High-frequency searches (post-merger BNS, glitch studies) |
| **2048 Hz** (downsampled) | 1024 Hz | Low-latency / matched filter at reduced cost |
| **512 Hz** (downsampled) | 256 Hz | Low-frequency studies (Schumann resonance, seismic) |

Downsampling reduces computational cost and file sizes. Upsampling is generally avoided in GW analysis (it adds no real information).

### 4.2 Downsampling with `resample()`

GWpy applies a low-pass anti-aliasing filter automatically before decimating:

In [ ]:
# Cell 9 — Downsampling demonstration
print(f"Original sample rate : {ts.sample_rate}")

ts_2048 = ts.resample(2048)
ts_1024 = ts.resample(1024)
ts_512  = ts.resample(512)

for label, data in [("4096 Hz", ts), ("2048 Hz", ts_2048),
                     ("1024 Hz", ts_1024), ("512 Hz", ts_512)]:
    print(f"  {label:10s} → {len(data):6d} samples, "
          f"Nyquist = {int(data.sample_rate.value)//2} Hz")

### 4.3 Aliasing and the Nyquist Theorem

The **Nyquist–Shannon sampling theorem** states:

> A band-limited signal with maximum frequency $f_{\max}$ can be perfectly reconstructed from samples taken at rate $f_s \geq 2 f_{\max}$.

The **Nyquist frequency** $f_N = f_s / 2$ is the highest frequency representable at rate $f_s$. Any signal component at $f > f_N$ will be **aliased** — it folds back into the band as a phantom frequency component at $f_N - (f - f_N) = 2f_N - f$.

For GW analysis with the 4096 Hz channel:
- $f_N = 2048$ Hz
- Frequencies above 2048 Hz in the raw analogue data would alias if not filtered first
- GWOSC data has already been conditioned; no aliasing occurs in the provided files

### 4.4 Anti-Aliasing Filters

In [ ]:
# Cell 10 — Visualising the effect of downsampling on the spectrum
from scipy.signal import welch

fig, ax = plt.subplots(figsize=(11, 4))

colors = ["#4575b4", "#74add1", "#f46d43", "#a50026"]
datasets = [
    (ts,      "4096 Hz (original)"),
    (ts_2048, "2048 Hz (resampled)"),
    (ts_1024, "1024 Hz (resampled)"),
    (ts_512,  " 512 Hz (resampled)"),
]

for (data, label), color in zip(datasets, colors):
    fs_i = int(data.sample_rate.value)
    freqs, psd = welch(data.value, fs=fs_i, nperseg=min(4*fs_i, len(data)))
    mask = freqs > 0
    ax.loglog(freqs[mask], np.sqrt(psd[mask]), lw=1.2, label=label, color=color)

ax.set_xlim(10, 3000)
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel(r"ASD [$\mathrm{strain}/\sqrt{\mathrm{Hz}}$]")
ax.set_title("GW150914 H1 — ASD at different sample rates")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("Note: the anti-aliasing filter rolls off the spectrum near each Nyquist frequency.")

---
## 5. Spectral Analysis

### 5.1 The Discrete Fourier Transform

The **Discrete Fourier Transform (DFT)** decomposes a time series $x[n]$ ($n = 0, \ldots, N-1$) into complex frequency components:

$$
X[k] = \sum_{n=0}^{N-1} x[n]\, e^{-2\pi i k n / N}, \quad k = 0, 1, \ldots, N-1
$$

The corresponding frequencies are $f_k = k \, f_s / N$, and the frequency resolution is
$$
\Delta f = \frac{f_s}{N} = \frac{1}{T}
$$
where $T = N/f_s$ is the duration of the segment. Longer segments → finer frequency resolution.

The **Fast Fourier Transform (FFT)** is the $\mathcal{O}(N \log N)$ algorithm used in practice. In Python: `numpy.fft.rfft()` for real-valued signals.

### 5.2 Power Spectral Density and Welch's Method

A single FFT of the full data segment gives a **periodogram**, which is a noisy, inconsistent estimator of the true spectrum. **Welch's method** improves it by:
1. Dividing the data into overlapping sub-segments
2. Windowing each sub-segment (e.g., Hann window) to suppress spectral leakage
3. Computing the periodogram of each sub-segment
4. Averaging the periodograms

The result is the **Power Spectral Density** (PSD), $S_n(f)$, with units of strain²/Hz.

GWpy provides `TimeSeries.psd()` which wraps `scipy.signal.welch`:

In [ ]:
# Cell 11 — Fetch a longer stretch for a reliable PSD estimate
# Use 512 s centred on the event; the signal itself is tiny and barely affects the PSD
h1_long = TimeSeries.fetch_open_data(
    "H1",
    GW150914_GPS - 256,
    GW150914_GPS + 256,
    sample_rate=4096,
    cache=True,
)
print(f"Loaded {float(h1_long.duration):.0f} s of data ({len(h1_long)} samples)")

In [ ]:
# Cell 12 — Compute the PSD using Welch's method
# fftlength=4 s → frequency resolution Δf = 1/4 = 0.25 Hz
psd_h1 = h1_long.psd(fftlength=4, overlap=2, window="hann")

print(f"PSD type       : {type(psd_h1).__name__}")
print(f"Frequency bins : {len(psd_h1)}")
print(f"Frequency range: {psd_h1.frequencies[0]:.3f} – {psd_h1.frequencies[-1]:.1f} Hz")
print(f"Δf             : {float(psd_h1.df):.4f} Hz")
print(f"PSD at 100 Hz  : {psd_h1.value[psd_h1.frequencies.value.searchsorted(100)]:.3e} strain²/Hz")

### 5.3 Amplitude Spectral Density

The **Amplitude Spectral Density** (ASD) is simply $\sqrt{S_n(f)}$, with units of strain/$\sqrt{\text{Hz}}$. It is the most commonly plotted quantity because it makes the noise floor directly comparable to signal strain amplitudes.

In [ ]:
# Cell 13 — Plot the ASD for H1 and L1
l1_long = TimeSeries.fetch_open_data(
    "L1",
    GW150914_GPS - 256,
    GW150914_GPS + 256,
    sample_rate=4096,
    cache=True,
)
psd_l1 = l1_long.psd(fftlength=4, overlap=2, window="hann")

fig, ax = plt.subplots(figsize=(11, 5))

for psd, label, color in [
    (psd_h1, "H1 (Hanford)",   "#4575b4"),
    (psd_l1, "L1 (Livingston)", "#d73027"),
]:
    freqs = psd.frequencies.value
    asd   = np.sqrt(psd.value)
    mask  = (freqs > 10) & (freqs < 2000)
    ax.loglog(freqs[mask], asd[mask], lw=1.0, label=label, color=color)

# Annotate key noise features
ax.axvline(60,  color="grey", lw=0.8, ls=":", alpha=0.8, label="60 Hz power line")
ax.axvline(120, color="grey", lw=0.8, ls=":", alpha=0.8)
ax.axvline(180, color="grey", lw=0.8, ls=":", alpha=0.8)
ax.annotate("60 Hz",  xy=(60,  2e-22), fontsize=8, color="grey", ha="center")
ax.annotate("120 Hz", xy=(120, 2e-22), fontsize=8, color="grey", ha="center")

ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel(r"ASD [$\mathrm{strain}/\sqrt{\mathrm{Hz}}$]")
ax.set_title("H1 and L1 Amplitude Spectral Density — GW150914 epoch")
ax.set_xlim(20, 2000)
ax.set_ylim(1e-24, 1e-19)
ax.legend()
plt.tight_layout()
plt.show()

### 5.4 Spectral Lines and Noise Features

The ASD reveals the rich structure of the instrument noise:

| Feature | Frequency | Origin |
|---|---|---|
| Seismic wall | < 20 Hz | Ground motion coupling |
| Thermal noise floor | 20–300 Hz | Suspension thermal noise |
| Shot noise rise | > 300 Hz | Photon counting statistics |
| Power line fundamental | 60 Hz (US) | AC electrical power |
| Power line harmonics | 120, 180, 240 Hz | Harmonics of above |
| Violin modes | ~500 Hz | Resonances of test-mass suspension wires |
| 1 Hz combs | Multiples of 1 Hz | Electronics, GPS timing |

These spectral lines are **non-stationary** — they can appear, disappear, or drift in frequency over time. Dealing with them is a key challenge in GW data analysis.

---
## 6. Band-Pass Filtering

### 6.1 Filter Design Concepts

A **band-pass filter** passes frequencies within a chosen band $[f_{\rm low}, f_{\rm high}]$ and attenuates everything outside it. Common design choices:

| Filter type | Characteristics |
|---|---|
| **Butterworth** | Maximally flat passband, monotone roll-off. Simple and widely used. |
| **Chebyshev I** | Equiripple in passband, steeper roll-off than Butterworth. |
| **Chebyshev II** | Equiripple in stopband, flat passband. |
| **Elliptic (Cauer)** | Equiripple in both bands; steepest roll-off for given order. |

For quick GW signal visualisation, a **4th-order Butterworth band-pass** from 30 Hz to 400 Hz is the community standard — it passes the GW chirp while cutting out the seismic wall and most of the thermal noise rise.

**Filter order** determines the steepness of the roll-off: a $n^{\rm th}$-order filter attenuates at $-6n$ dB/octave beyond the corner frequency.

### 6.2 Butterworth Band-Pass Filter

GWpy wraps `scipy.signal.butter` in the `bandpass()` method:

In [ ]:
# Cell 14 — Apply a Butterworth band-pass filter
# We apply to the 32-second segment around GW150914

bp_h1 = h1_150914.bandpass(30, 400, filtfilt=True)

print(f"Input  length: {len(h1_150914)} samples")
print(f"Output length: {len(bp_h1)} samples (same — band-pass preserves length)")
print(f"Peak strain after band-pass: {float(np.max(np.abs(bp_h1.value))):.3e}")

In [ ]:
# Cell 15 — Compare raw and band-passed strain around GW150914
# Zoom in to ±0.5 s around the merger
t0 = GW150914_GPS
raw_crop = h1_150914.crop(t0 - 0.5, t0 + 0.5)
bp_crop  = bp_h1.crop(t0 - 0.5, t0 + 0.5)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

t_rel_raw = raw_crop.times.value - t0
t_rel_bp  = bp_crop.times.value  - t0

axes[0].plot(t_rel_raw, raw_crop.value * 1e21, lw=0.5, color="#4575b4", alpha=0.8)
axes[0].set_ylabel(r"Raw strain [$\times 10^{-21}$]")
axes[0].set_title("GW150914 H1 — raw strain (no filtering)")

axes[1].plot(t_rel_bp, bp_crop.value * 1e21, lw=1.2, color="#d73027")
axes[1].set_ylabel(r"Band-passed [$\times 10^{-21}$]")
axes[1].set_title("GW150914 H1 — band-pass 30–400 Hz (4th-order Butterworth)")
axes[1].set_xlabel("Time relative to merger [s]")

for ax in axes:
    ax.axvline(0, color="k", lw=1.0, ls="--", alpha=0.5)

plt.tight_layout()
plt.show()

The band-pass removes the dominant low-frequency seismic noise and the high-frequency shot noise, but the signal is still hard to see. The remaining noise in-band is still much larger than the signal. **Whitening** will fix this.

### 6.3 Edge Effects and Windowing

IIR (Infinite Impulse Response) filters — like Butterworth — have **filter transients** at the start and end of the time series. These are artefacts introduced by the filter's initial conditions.

**`filtfilt=True`** in GWpy applies the filter forward *and* backward, which:
- Doubles the effective filter order
- Eliminates phase distortion (zero-phase filtering)
- Still has edge transients — the first and last few seconds should be cropped away

A rule of thumb: remove **4 × filter_order / sample_rate** seconds from each end after filtering.

In [ ]:
# Cell 16 — Demonstrate edge effects of band-pass filtering
fig, ax = plt.subplots(figsize=(12, 3))
t_rel_full = bp_h1.times.value - float(bp_h1.t0)
ax.plot(t_rel_full, bp_h1.value * 1e21, lw=0.5, color="#4575b4", alpha=0.9)

# Shade the edge transient regions (first and last 1 s)
ax.axvspan(0, 1.0,  alpha=0.15, color="red", label="edge transient region (discard)")
ax.axvspan(float(bp_h1.duration.value)-1.0, float(bp_h1.duration.value),
           alpha=0.15, color="red")

ax.set_xlabel("Time from data start [s]")
ax.set_ylabel(r"Strain [$\times 10^{-21}$]")
ax.set_title("Band-pass filter edge transients (should be discarded)")
ax.legend()
plt.tight_layout()
plt.show()

### 6.4 Notch Filters for Spectral Lines

Narrow spectral lines (power line harmonics, violin modes) can be removed with **notch filters** — band-stop filters that reject a very narrow frequency band:

In [ ]:
# Cell 17 — Apply notch filters to suppress 60 Hz and harmonics
from scipy.signal import iirnotch, sosfilt, zpk2sos
from gwpy.signal import notch as gwpy_notch

# GWpy's notch() method applies a notch filter at a given frequency
notch_freqs = [60.0, 120.0, 180.0]  # Hz
h1_notched = h1_150914.copy()
for f0 in notch_freqs:
    h1_notched = h1_notched.notch(f0, filtfilt=True)

# Compare PSDs before and after notching
psd_orig    = h1_150914.psd(fftlength=4, overlap=2, window="hann")
psd_notched = h1_notched.psd(fftlength=4, overlap=2, window="hann")

fig, ax = plt.subplots(figsize=(11, 4))
f_orig    = psd_orig.frequencies.value
f_notched = psd_notched.frequencies.value
mask = (f_orig > 20) & (f_orig < 500)

ax.semilogy(f_orig[mask],    np.sqrt(psd_orig.value[mask]),    lw=0.8, label="Original",       color="#4575b4")
ax.semilogy(f_notched[mask], np.sqrt(psd_notched.value[mask]), lw=0.8, label="Notched (60/120/180 Hz)", color="#d73027")
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel(r"ASD [strain/$\sqrt{\rm Hz}$]")
ax.set_title("Effect of notch filters on the ASD")
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Whitening

### 7.1 The Noise Floor Problem

Even after band-passing, the noise is still **coloured** — it has much more power at low frequencies (~30–100 Hz) than at higher frequencies (~200–400 Hz). This means:

1. A GW signal at 150 Hz is harder to see than the noise at 50 Hz
2. Matched filtering (the optimal detection algorithm) implicitly weights by $1/S_n(f)$ to compensate — effectively whitening in the frequency domain
3. For *visualisation*, we whiten explicitly so the eye can see the signal against a flat noise floor

### 7.2 Theoretical Foundation

**Whitening** is a linear transformation that makes the noise power uniform across all frequencies:

$$
\tilde{x}_{\rm white}(f) = \frac{\tilde{x}(f)}{\sqrt{S_n(f)}}
$$

where $\tilde{x}(f)$ is the Fourier transform of the data and $S_n(f)$ is the one-sided PSD. After this operation, the noise in every frequency bin has unit variance (in a statistical sense).

The algorithm in practice:
1. Estimate the PSD $S_n(f)$ using Welch's method on a **background** segment (ideally not containing the signal)
2. Fourier-transform the data segment of interest
3. Divide by $\sqrt{S_n(f)}$ in the frequency domain
4. Inverse-Fourier-transform back to the time domain
5. Band-pass filter to suppress the tails (where the PSD estimate may be unreliable)

### 7.3 Whitening with GWpy

GWpy provides `TimeSeries.whiten()` which automates all the steps above:

In [ ]:
# Cell 18 — Whiten the GW150914 H1 strain
# fftlength: length of each PSD sub-segment (in seconds)
# overlap  : overlap between sub-segments (in seconds)
# fduration: length of the whitening filter in the time domain (in seconds)
#            longer → sharper frequency resolution but more edge artefacts

h1_white = h1_150914.whiten(
    fftlength=4,
    overlap=2,
    method="median",
    fduration=2,
    window="hann",
)

# After whitening, the data is dimensionless (strain/ASD * sqrt(Hz))
print(f"Whitened data type: {type(h1_white).__name__}")
print(f"std of whitened data: {float(np.std(h1_white.value)):.3f}  (should be ≈ 1.0 for Gaussian noise)")
print(f"Peak whitened value: {float(np.max(np.abs(h1_white.value))):.2f} σ")

> **Key insight:** The standard deviation of whitened noise is ≈ 1.0, so a whitened GW signal with peak value ~5 has a signal-to-noise ratio of ≈ 5. GW150914 had a network SNR of ~24 (combined H1 + L1).

### 7.4 Whitening Parameters and Trade-offs

In [ ]:
# Cell 19 — Effect of fftlength on whitening quality
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

t0 = GW150914_GPS
fftlengths = [1, 4, 16]
colors = ["#d73027", "#4575b4", "#1a9850"]

for ax, ffl, color in zip(axes, fftlengths, colors):
    w = h1_150914.whiten(fftlength=ffl, overlap=ffl//2, fduration=ffl/2, window="hann")
    wc = w.crop(t0 - 0.5, t0 + 0.5)
    t_rel = wc.times.value - t0
    ax.plot(t_rel, wc.value, lw=0.8, color=color)
    ax.axvline(0, color="k", lw=1.0, ls="--", alpha=0.6)
    ax.set_ylabel(r"Whitened strain [$\sigma$]")
    ax.set_title(f"fftlength = {ffl} s  (Δf = {1/ffl:.3f} Hz)")
    ax.set_ylim(-8, 8)

axes[-1].set_xlabel("Time relative to GW150914 [s]")
fig.suptitle("Effect of PSD segment length on whitening", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

**Trade-off summary:**

| `fftlength` | Δf (Hz) | Frequency resolution | Edge transients | Best for |
|---|---|---|---|---|
| 1 s | 1.0 Hz | Coarse — misses narrow lines | Short | Quick look |
| 4 s | 0.25 Hz | Good for most purposes | ~2 s each side | Standard GW analysis |
| 16 s | 0.063 Hz | Resolves fine spectral structure | ~8 s each side | Narrow-line searches |

---
## 8. Visualisation — GW150914

### 8.1 Raw vs Band-Passed Strain

We now have all the tools to produce the canonical GW150914 visualisation: three panels showing the progressive effect of signal processing.

In [ ]:
# Cell 20 — Fetch L1 data for GW150914
l1_150914 = TimeSeries.fetch_open_data(
    "L1",
    GW150914_GPS - 16,
    GW150914_GPS + 16,
    sample_rate=4096,
    cache=True,
)
print(f"L1 data loaded: {float(l1_150914.duration):.1f} s")

In [ ]:
# Cell 21 — GW150914: raw vs band-passed vs whitened (H1 and L1)
t0 = GW150914_GPS
twindow = 0.4  # seconds to show around merger

# Process H1
h1_bp = h1_150914.bandpass(30, 400, filtfilt=True)
h1_wh = h1_150914.whiten(fftlength=4, overlap=2, fduration=2, window="hann")
h1_wh_bp = h1_wh.bandpass(30, 400, filtfilt=True)  # whiten + band-pass

# Process L1 — note: L1 signal is inverted relative to H1 (opposite arm orientation)
# We flip L1 for comparison
l1_bp = l1_150914.bandpass(30, 400, filtfilt=True)
l1_wh = l1_150914.whiten(fftlength=4, overlap=2, fduration=2, window="hann")
l1_wh_bp = l1_wh.bandpass(30, 400, filtfilt=True)

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
fig.suptitle("GW150914 — Progressive Signal Processing", fontsize=14, fontweight="bold")

ifo_data = {
    "H1": {"raw": h1_150914, "bp": h1_bp, "wh": h1_wh_bp, "color": "#4575b4", "sign": 1},
    "L1": {"raw": l1_150914, "bp": l1_bp, "wh": l1_wh_bp, "color": "#d73027", "sign": -1},
}

row_labels = ["Raw strain", "Band-pass (30–400 Hz)", "Whitened + Band-pass"]
row_keys   = ["raw", "bp", "wh"]
row_units  = [r"$\times 10^{-21}$", r"$\times 10^{-21}$", r"$\sigma$ (whitened units)"]

for col, (ifo, d) in enumerate(ifo_data.items()):
    for row, (key, unit) in enumerate(zip(row_keys, row_units)):
        ax = axes[row, col]
        ts_crop = d[key].crop(t0 - twindow, t0 + twindow)
        t_rel   = ts_crop.times.value - t0
        scale   = 1e21 if key != "wh" else 1.0
        ax.plot(t_rel, d["sign"] * ts_crop.value * scale,
                lw=0.8, color=d["color"])
        ax.axvline(0, color="k", lw=1.0, ls="--", alpha=0.5)
        if col == 0:
            ax.set_ylabel(f"{row_labels[row]}\n[{unit}]", fontsize=9)
        if row == 0:
            ax.set_title(f"{ifo} ({'Hanford' if ifo=='H1' else 'Livingston'})",
                         fontsize=12, fontweight="bold")
        if row == 2:
            ax.set_xlabel("Time relative to merger [s]")

plt.tight_layout()
plt.show()

**What we see:**
- **Row 1 (raw):** Pure noise — no signal visible
- **Row 2 (band-pass):** Low-frequency noise is reduced, but the GW chirp is still hidden
- **Row 3 (whitened + band-pass):** The GW150914 chirp is clearly visible in both detectors — they see the same signal 7 ms apart (the light-travel time between Hanford and Livingston)

### 8.2 Whitened Strain

In [ ]:
# Cell 22 — Close-up of the whitened GW150914 signal
fig, ax = plt.subplots(figsize=(12, 4))
t0 = GW150914_GPS

for ifo, data, color, sign in [
    ("H1", h1_wh_bp, "#4575b4",  1),
    ("L1", l1_wh_bp, "#d73027", -1),
]:
    wc = data.crop(t0 - 0.35, t0 + 0.15)
    t_rel = wc.times.value - t0
    ax.plot(t_rel, sign * wc.value, lw=1.5, color=color, label=ifo)

ax.axvline(0,       color="k", lw=1.0, ls="--", alpha=0.6, label="H1 merger")
ax.axvline(-0.007,  color="k", lw=0.8, ls=":",  alpha=0.6, label="L1 merger (7 ms earlier)")
ax.set_xlabel("Time relative to H1 merger [s]")
ax.set_ylabel(r"Whitened strain [$\sigma$]")
ax.set_title("GW150914 — Whitened strain: H1 and L1 overlaid")
ax.legend()
plt.tight_layout()
plt.show()

### 8.3 Q-Transform of the Processed Data

The **Q-transform** is a time–frequency representation optimised for transient signals. It uses a Gaussian window whose width (in time and frequency) is controlled by the *quality factor* $Q$. High $Q$ → good frequency resolution; low $Q$ → good time resolution.

In [ ]:
# Cell 23 — Q-transform of GW150914 (H1)
# Use the full 32-second segment for better statistics, zoom in the plot
t0 = GW150914_GPS

# White the data first for better contrast
h1_for_qt = h1_150914.whiten(fftlength=4, overlap=2, fduration=2)

# Compute Q-transform
qt = h1_for_qt.q_transform(
    qrange=(8, 32),       # range of Q values to scan
    frange=(20, 500),     # frequency range in Hz
    gps=t0,               # GPS time to centre the search
    search=0.5,           # seconds to search around GPS time for best Q
    tres=0.002,           # time resolution in s
    fres=0.5,             # frequency resolution in Hz
    whiten=False,         # already whitened
    outseg=(t0 - 0.5, t0 + 0.5),  # output segment
)

fig = plt.figure(figsize=(11, 5))
ax = fig.add_subplot(1, 1, 1)
pc = ax.pcolormesh(
    qt.times.value - t0,
    qt.frequencies.value,
    qt.value.T,
    norm=LogNorm(vmin=1, vmax=25),
    cmap="viridis",
    shading="auto",
)
fig.colorbar(pc, ax=ax, label=r"Normalized energy [$\sigma$]")
ax.set_yscale("log")
ax.set_xlabel("Time relative to GW150914 [s]")
ax.set_ylabel("Frequency [Hz]")
ax.set_title("GW150914 H1 — Q-transform (whitened)")
ax.axvline(0, color="w", lw=1.2, ls="--", alpha=0.8)
plt.tight_layout()
plt.show()

The Q-transform reveals the **gravitational-wave chirp** of GW150914: a track that sweeps from ~35 Hz up to ~150 Hz in under 0.2 seconds, ending abruptly at the moment of merger. The **ringdown** — where the newly formed black hole rings down like a struck bell — is faintly visible as a short tail immediately after the merger.

---
## 9. Visualisation — GW170817

### 9.1 Characteristics of a Binary Neutron Star Signal

GW170817 (17 August 2017) was the first binary neutron-star merger observed by LIGO and Virgo. Neutron stars are much less massive than the GW150914 black holes (~1.4 $M_\odot$ each vs. ~30–35 $M_\odot$), which has profound consequences:

| Property | GW150914 (BBH) | GW170817 (BNS) |
|---|---|---|
| Component masses | ~36 + 29 $M_\odot$ | ~1.4 + 1.2 $M_\odot$ |
| Total mass | ~65 $M_\odot$ | ~2.8 $M_\odot$ |
| In-band chirp duration | ≲ 0.2 s | ≈ 100 s |
| Merger frequency | ~150 Hz | ~1500 Hz |
| Luminosity distance | ~440 Mpc | ~40 Mpc |
| Peak strain | ~$10^{-21}$ | ~$10^{-22}$ |
| Network SNR | ~24 | ~33 |

The BNS chirp is **much longer** (100 s in-band) but **fainter** per unit time. This makes whitening and visualisation more challenging — we need a longer background segment.

In [ ]:
# Cell 24 — Fetch H1 data around GW170817 (need a longer stretch)
GW170817_GPS = 1187008882.4

# Fetch 128 s around the merger (the chirp starts ~100 s before merger)
h1_170817 = TimeSeries.fetch_open_data(
    "H1",
    GW170817_GPS - 120,
    GW170817_GPS + 8,
    sample_rate=4096,
    cache=True,
)
l1_170817 = TimeSeries.fetch_open_data(
    "L1",
    GW170817_GPS - 120,
    GW170817_GPS + 8,
    sample_rate=4096,
    cache=True,
)

print(f"GW170817 GPS: {GW170817_GPS}")
print(f"UTC:         {from_gps(GW170817_GPS)}")
print(f"H1 loaded  : {float(h1_170817.duration):.0f} s")
print(f"L1 loaded  : {float(l1_170817.duration):.0f} s")

### 9.2 Raw vs Whitened Strain for GW170817

In [ ]:
# Cell 25 — Whiten GW170817 data
# Use a longer fftlength for the BNS (better low-frequency PSD estimate)
h1_170817_wh = h1_170817.whiten(fftlength=4, overlap=2, fduration=2, window="hann")
l1_170817_wh = l1_170817.whiten(fftlength=4, overlap=2, fduration=2, window="hann")

# Band-pass to 30–400 Hz to see the main chirp track
h1_170817_wh_bp = h1_170817_wh.bandpass(30, 400, filtfilt=True)
l1_170817_wh_bp = l1_170817_wh.bandpass(30, 400, filtfilt=True)

print(f"H1 whitened std: {float(np.std(h1_170817_wh.value)):.3f}")

In [ ]:
# Cell 26 — GW170817: raw vs whitened H1 strain
t0 = GW170817_GPS

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Raw strain (last 30 s before merger)
raw_crop = h1_170817.crop(t0 - 30, t0 + 2)
wh_crop  = h1_170817_wh_bp.crop(t0 - 30, t0 + 2)

t_raw = raw_crop.times.value - t0
t_wh  = wh_crop.times.value  - t0

axes[0].plot(t_raw, raw_crop.value * 1e21, lw=0.4, color="#4575b4")
axes[0].set_ylabel(r"Raw strain H1 [$\times 10^{-21}$]")
axes[0].set_title("GW170817 H1 — raw strain (last 30 s before merger)")

axes[1].plot(t_wh, wh_crop.value, lw=0.6, color="#d73027")
axes[1].set_ylabel(r"Whitened strain H1 [$\sigma$]")
axes[1].set_title("GW170817 H1 — whitened + band-pass 30–400 Hz")
axes[1].set_xlabel("Time before merger [s]")

for ax in axes:
    ax.axvline(0, color="k", lw=1.0, ls="--", alpha=0.6, label="merger")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Cell 27 — GW170817 Q-transform: the long BNS chirp
t0 = GW170817_GPS

# Use a shorter window for the Q-transform output for clarity
qt_170817 = h1_170817_wh.q_transform(
    qrange=(8, 64),
    frange=(20, 500),
    gps=t0,
    search=0.5,
    tres=0.02,
    fres=0.5,
    whiten=False,
    outseg=(t0 - 30, t0 + 2),
)

fig, ax = plt.subplots(figsize=(13, 5))
pc = ax.pcolormesh(
    qt_170817.times.value - t0,
    qt_170817.frequencies.value,
    qt_170817.value.T,
    norm=LogNorm(vmin=1, vmax=15),
    cmap="viridis",
    shading="auto",
)
fig.colorbar(pc, ax=ax, label=r"Normalized energy [$\sigma$]")
ax.set_yscale("log")
ax.set_xlabel("Time relative to GW170817 merger [s]")
ax.set_ylabel("Frequency [Hz]")
ax.set_title("GW170817 H1 — Q-transform: the binary neutron star chirp")
ax.axvline(0, color="w", lw=1.2, ls="--", alpha=0.8, label="merger")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### 9.3 Comparing GW150914 and GW170817 in the Time Domain

In [ ]:
# Cell 28 — Side-by-side whitened Q-transforms: BBH vs BNS
# Use the pre-computed qt (GW150914) and qt_170817 (GW170817)
t0_150914 = GW150914_GPS
t0_170817 = GW170817_GPS

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# GW150914 (already computed)
axes[0].pcolormesh(
    qt.times.value - t0_150914,
    qt.frequencies.value,
    qt.value.T,
    norm=LogNorm(vmin=1, vmax=25),
    cmap="viridis",
    shading="auto",
)
axes[0].set_yscale("log")
axes[0].set_title("GW150914 (BBH, ~30+29 M☉)\nDuration: ~0.2 s", fontsize=11)
axes[0].set_xlabel("Time relative to merger [s]")
axes[0].set_ylabel("Frequency [Hz]")
axes[0].set_xlim(-0.5, 0.5)

# GW170817 (last 30 s before merger)
axes[1].pcolormesh(
    qt_170817.times.value - t0_170817,
    qt_170817.frequencies.value,
    qt_170817.value.T,
    norm=LogNorm(vmin=1, vmax=15),
    cmap="viridis",
    shading="auto",
)
axes[1].set_yscale("log")
axes[1].set_title("GW170817 (BNS, ~1.4+1.2 M☉)\nDuration: ~100 s", fontsize=11)
axes[1].set_xlabel("Time relative to merger [s]")
axes[1].set_ylabel("Frequency [Hz]")

for ax in axes:
    ax.axvline(0, color="w", lw=1.2, ls="--", alpha=0.8)

fig.suptitle("Binary Black Hole vs Binary Neutron Star: Q-transform comparison",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 10. Handling Gaps, Data Artifacts, and GPS Time Conventions

### 10.1 Understanding Data Gaps

Real detector data is not continuous. The interferometer regularly enters and exits **Science mode** (also called *Observing mode*) due to:

- **Lock losses** — the interferometer loses its optical resonance and must be relocked
- **Planned maintenance** — commissioning, mirror alignment, vacuum work
- **Environmental disturbances** — earthquakes, wind, anthropogenic noise
- **Electronics or controls issues**

A gap in the data is not just missing samples — it means the `TimeSeries` has a discontinuity. GWpy represents this with the `StateTimeSeries` and `DataQualityFlag` objects, and sometimes returns a `TimeSeriesDict` with multiple segments.

In [ ]:
# Cell 29 — Inspect data gaps using gwosc.timeline
from gwosc.timeline import get_segments

# Get the H1 Science-mode segments during a 1-hour window around GW150914
t_start = GW150914_GPS - 1800  # 30 min before
t_end   = GW150914_GPS + 1800  # 30 min after

segs = get_segments("H1_DATA", t_start, t_end)

print(f"H1 Science-mode segments in ±30 min around GW150914:")
total_sci = 0
for seg in segs:
    duration = seg[1] - seg[0]
    total_sci += duration
    print(f"  [{seg[0]:.0f}, {seg[1]:.0f}]  duration = {duration:.1f} s")

print(f"\nTotal Science-mode time: {total_sci:.1f} s / {t_end - t_start:.1f} s")
print(f"Duty cycle: {100*total_sci/(t_end - t_start):.1f}%")

### 10.2 Gap Filling Strategies

When a gap falls within an analysis segment, you have several options:

| Strategy | Use case | Pros | Cons |
|---|---|---|---|
| **Truncate** | Signal well before/after gap | Clean — no artefacts | Loses data |
| **Zero-pad** | Spectral analysis | Simple | Introduces discontinuity |
| **Mean/median fill** | Visualisation only | Reduces jump | Biases statistics |
| **Gate and inpaint** | Glitch mitigation | Preserves Gaussianity | Complex |
| **Reject the segment** | Short gaps near signal | Conservative | Loses data |

For **signal searches**, the standard approach is to avoid gaps entirely by selecting only continuous Science-mode segments longer than the analysis window.

In [ ]:
# Cell 30 — Visualise the Science-mode segments around GW150914
fig, ax = plt.subplots(figsize=(13, 2.5))

for seg in segs:
    t_s = seg[0] - GW150914_GPS
    t_e = seg[1] - GW150914_GPS
    ax.barh(0.5, t_e - t_s, left=t_s, height=0.6,
            color="#1a9850", edgecolor="k", lw=0.5, alpha=0.8)

ax.axvline(0, color="crimson", lw=2, label="GW150914")
ax.set_xlim((t_start - GW150914_GPS), (t_end - GW150914_GPS))
ax.set_xlabel("Time relative to GW150914 [s]")
ax.set_yticks([])
ax.set_title("H1 Science-mode segments ±30 min around GW150914")
ax.legend()
plt.tight_layout()
plt.show()

### 10.3 Common Data Artifacts

Beyond gaps, real data contains a variety of **glitches** — transient noise events that can mimic or obscure GW signals:

| Glitch type | Description | Typical duration | Typical frequency |
|---|---|---|---|
| **Blip** | Short, broadband transient | 10–100 ms | 30–300 Hz |
| **Koi fish** | Long, chirp-like artefact | ~1 s | 10–100 Hz |
| **Scattered light** | Arch-shaped tracks in Q-transform | 1–10 s | 20–100 Hz |
| **Violin mode** | Narrow-band resonance | minutes | ~500 Hz |
| **Power line excess** | 60/50 Hz and harmonics | continuous | 60 Hz, 120 Hz, ... |
| **Gravity Spy glitches** | Many morphological types | varied | varied |

The **Gravity Spy** project (https://www.gravityspy.org) uses machine learning to classify glitches in LIGO data. Students can contribute to the classification effort!

In [ ]:
# Cell 31 — Illustrate glitch identification in the Q-transform
# Fetch a random quiet stretch with a known glitch for comparison
# We use a stretch from O1 known to contain scattered-light artefacts

# For demonstration, we show a segment from the H1 data that
# is NOT around GW150914 — a random 32-second stretch
t_glitch_demo = GW150914_GPS - 200  # 200 s before the event

h1_demo = TimeSeries.fetch_open_data(
    "H1",
    t_glitch_demo - 16,
    t_glitch_demo + 16,
    sample_rate=4096,
    cache=True,
)

h1_demo_wh = h1_demo.whiten(fftlength=4, overlap=2, fduration=2)

qt_demo = h1_demo_wh.q_transform(
    qrange=(4, 64),
    frange=(10, 1024),
    gps=t_glitch_demo,
    search=8,
    tres=0.002,
    fres=0.5,
    whiten=False,
    outseg=(t_glitch_demo - 8, t_glitch_demo + 8),
)

fig, ax = plt.subplots(figsize=(12, 5))
pc = ax.pcolormesh(
    qt_demo.times.value - t_glitch_demo,
    qt_demo.frequencies.value,
    qt_demo.value.T,
    norm=LogNorm(vmin=1, vmax=10),
    cmap="plasma",
    shading="auto",
)
fig.colorbar(pc, ax=ax, label=r"Normalized energy [$\sigma$]")
ax.set_yscale("log")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Frequency [Hz]")
ax.set_title("H1 Q-transform — 200 s before GW150914 (background noise only)")
plt.tight_layout()
plt.show()

### 10.4 GPS Time Conventions in Practice

GPS time is the universal time standard in GW astronomy. Key facts:

1. **GPS epoch**: 00:00:00 UTC on 6 January 1980 = GPS 0.0
2. **No leap seconds**: GPS time does not skip seconds (unlike UTC). As of 2024, GPS is ahead of UTC by **18 seconds**.
3. **TAI relationship**: GPS = TAI − 19 s (TAI is International Atomic Time)
4. **Precision**: LIGO timestamps are accurate to < 1 μs
5. **Relative time in plots**: Always plot time relative to an event GPS time for publication

In [ ]:
# Cell 32 — GPS time manipulation examples
from gwpy.time import to_gps, from_gps
from astropy.time import Time

# Key event GPS times
events = {
    "GW150914": 1126259462.4,
    "GW170817": 1187008882.4,
    "GW190521": 1242442967.4,
    "GW200105": 1262276512.0,
}

print(f"{'Event':<12} {'GPS time':>15} {'UTC date/time':>35} {'Days since GW150914':>22}")
print("-" * 90)
gps_ref = events["GW150914"]
for name, gps in events.items():
    utc = Time(gps, format="gps", scale="utc")
    days_since = (gps - gps_ref) / 86400.0
    print(f"{name:<12} {gps:>15.1f} {str(utc.iso):>35} {days_since:>22.1f}")

print()
print("GPS to UTC conversion using gwpy:")
print(f"  GW150914: {from_gps(GW150914_GPS)}")
print(f"  GW170817: {from_gps(GW170817_GPS)}")

In [ ]:
# Cell 33 — UTC to GPS conversion
from astropy.time import Time

# Convert a UTC string to GPS
utc_strings = [
    "2015-09-14 09:50:45",
    "2017-08-17 12:41:04",
    "2019-05-21 03:02:29",
]

print(f"{'UTC':>30} → {'GPS time':>20}")
print("-" * 55)
for utc_str in utc_strings:
    t = Time(utc_str, format="iso", scale="utc")
    gps = t.gps
    print(f"{utc_str:>30} → {gps:>20.1f}")

### 10.5 Segment Lists and the `DataQualityFlag`

A **segment list** is a set of non-overlapping time intervals during which some condition holds (e.g., the detector is in Science mode). GWpy has a rich segment management system:

In [ ]:
# Cell 34 — Working with segment lists
from gwpy.segments import Segment, SegmentList

# Build a simple SegmentList from the GWOSC Science-mode segments
segs_gwpy = SegmentList([Segment(s[0], s[1]) for s in segs])

print(f"Number of Science-mode segments: {len(segs_gwpy)}")
print(f"Total active time: {float(segs_gwpy.extent()[1] - segs_gwpy.extent()[0]):.1f} s (including gaps)")
print(f"Coalesced? {segs_gwpy == segs_gwpy.coalesce()}")

# Check if the GW150914 GPS time falls within a Science segment
gw_seg = Segment(GW150914_GPS, GW150914_GPS)  # point segment
in_science = any(GW150914_GPS in s for s in segs_gwpy)
print(f"\nGW150914 is in a Science-mode segment: {in_science}")

# Find which segment contains GW150914
for s in segs_gwpy:
    if s[0] <= GW150914_GPS <= s[1]:
        print(f"Containing segment: [{s[0]:.0f}, {s[1]:.0f}]  " +
              f"(duration = {s[1]-s[0]:.1f} s, event at +{GW150914_GPS-s[0]:.1f} s)")

---
## 11. Advanced Topics

### 11.1 The Q-Transform Revisited

The **Q-transform** (Constant-Q Transform, CQT) provides a time–frequency decomposition where the frequency resolution scales with frequency. This makes it ideal for GW signals, whose frequency evolves logarithmically (as $f \sim t^{-3/8}$ for a chirp).

The Q-transform of a signal $x(t)$ at time $\tau$, frequency $f_0$, and quality factor $Q$ is:

$$
X(\tau, f_0, Q) = \int_{-\infty}^{\infty} x(t)\, w\!\left(\frac{t-\tau}{f_0/Q}\right) e^{-2\pi i f_0 t} \, dt
$$

where $w$ is a normalised Gaussian window of width proportional to $Q/f_0$. The time–frequency resolution obeys:

$$\Delta t \sim \frac{Q}{2\pi f_0}, \qquad \Delta f \sim \frac{f_0}{Q}$$

Higher $Q$ → better frequency resolution but poorer time resolution. The standard Q-range for GW signals is $Q = 4$–$64$.

### 11.2 Stationarity and Non-Stationarity

All the analysis above assumes the noise is **stationary** — its statistical properties do not change with time. Real detector noise is **non-stationary** due to:

- Changing environmental conditions (wind, human activity, tides)
- Drifting alignment and gain settings
- Glitches and transients

Non-stationarity degrades the PSD estimate and hence the whitening quality. Mitigation strategies include:

1. **Short PSD segments** — use a recent estimate rather than a long average
2. **BayesLine** — a Bayesian PSD estimator that models spectral lines
3. **gating** — zero out loud glitches before computing the PSD
4. **PSD drift monitoring** — track how the ASD changes over time

In [ ]:
# Cell 35 — PSD stationarity check: compare PSD from different time chunks
chunk_dur = 32  # seconds per chunk
n_chunks  = int(float(h1_long.duration.value) // chunk_dur)

fig, ax = plt.subplots(figsize=(11, 5))
cmap = plt.cm.plasma

for i in range(n_chunks):
    t_start_i = float(h1_long.t0.value) + i * chunk_dur
    chunk = h1_long.crop(t_start_i, t_start_i + chunk_dur)
    psd_i = chunk.psd(fftlength=4, overlap=2, window="hann")
    freqs = psd_i.frequencies.value
    mask = (freqs > 20) & (freqs < 1000)
    color = cmap(i / n_chunks)
    ax.loglog(freqs[mask], np.sqrt(psd_i.value[mask]),
              lw=0.6, alpha=0.6, color=color,
              label=f"+{i*chunk_dur} s" if i in [0, n_chunks//2, n_chunks-1] else None)

ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel(r"ASD [strain/$\sqrt{\rm Hz}$]")
ax.set_title(f"H1 PSD stationarity check: {n_chunks} × {chunk_dur} s chunks\n"
             "(colour = time from start, blue→yellow)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 11.3 Online vs Offline Processing

GW data analysis operates in two modes:

| Mode | Latency | Purpose | PSD estimation |
|---|---|---|---|
| **Online (low-latency)** | Seconds to minutes | Early warning (EM follow-up) | Short, updated continuously |
| **Offline** | Hours to days | Final parameter estimation, catalogue | Long, carefully conditioned |

Low-latency pipelines such as **GstLAL**, **PyCBC Live**, and **MBTA** must whiten and filter data in real time, updating the PSD estimate every few seconds using a sliding window. The trade-off is a noisier PSD and hence slightly reduced sensitivity compared to offline analysis.

For this course, we work entirely in offline mode — we have the luxury of the full GWOSC data archive.

---
## 12. Student Exercises

The exercises below are graded from ⭐ (straightforward) to ⭐⭐⭐⭐ (challenging). They are designed to build on the concepts covered in this lesson.

---

### Exercise 1 — TimeSeries Basics ⭐

**Coding.** Load 32 seconds of L1 strain data centred on GW150914.

1. Print the GPS start time, duration, sample rate, and number of samples.
2. Convert the GPS start time to a UTC date and time string.
3. Compute the mean, standard deviation, and RMS of the raw strain.
4. Scale the strain to units of $10^{-21}$ and plot 2 seconds centred on the merger.
5. What is the maximum absolute strain value in the zoomed window? At what GPS time does it occur?

---

### Exercise 2 — The Nyquist Theorem and Aliasing ⭐⭐

**Conceptual and coding.**

1. Start with H1 data sampled at 4096 Hz. Compute and plot the ASD from 10 Hz to 2048 Hz.
2. Downsample to 2048 Hz (no anti-aliasing filter) by taking every other sample (`data[::2]`). Compute the ASD of this signal.
3. Downsample properly using `resample(2048)`. Compare the three ASDs on the same plot.
4. Explain in your own words: why does naive downsampling (step 2) distort the spectrum? What frequency bands are most affected?
5. For GW analysis, is 2048 Hz sufficient to capture the GW150914 signal? What is the highest frequency expected from the GW150914 merger?

---

### Exercise 3 — Band-Pass Filter Tuning ⭐⭐

**Coding.** Apply band-pass filters with different frequency ranges to H1 GW150914 data and compare the results.

1. Apply the following band-pass filters (4th-order Butterworth, `filtfilt=True`):
   - 30–400 Hz (standard)
   - 50–250 Hz (narrow)
   - 15–2000 Hz (very wide)
2. Plot the time-domain signal (±0.5 s around merger) for each filter on the same axes.
3. Plot the ASDs of the three filtered signals on a single log-log plot.
4. Which band-pass gives the clearest view of the GW150914 chirp? Justify your answer using the ASD of the unfiltered data.
5. What happens if you set the low-pass corner to 10 Hz instead of 30 Hz? Why might this make the signal *harder* to see?

---

### Exercise 4 — Whitening Diagnostics ⭐⭐

**Coding and analysis.** A well-whitened noise segment should be approximately Gaussian with zero mean and unit standard deviation.

1. Whiten 32 s of H1 data around GW150914 (with `fftlength=4`).
2. Crop out the edge transients (first and last 2 s).
3. Compute and print: mean, std, skewness, and excess kurtosis of the whitened data.
4. Plot a **histogram** of the whitened strain values. Overlay a standard normal distribution $\mathcal{N}(0, 1)$.
5. Compute the **Q–Q plot** (quantile–quantile plot) using `scipy.stats.probplot`. How Gaussian is the whitened noise?
6. *(Bonus)* Repeat steps 2–5 for the *unwhitened* band-passed strain. How does the distribution differ?

---

### Exercise 5 — GW170817: The Long Chirp ⭐⭐⭐

**Coding.** GW170817 is challenging because the chirp lasts ~100 s in the LIGO band.

1. Fetch 128 s of H1 and L1 data ending at the GW170817 merger time.
2. Whiten both detectors (use `fftlength=4`).
3. Apply a band-pass filter from 25 Hz to 500 Hz.
4. Plot the whitened, band-passed strain for both detectors over the full 128 s.
5. Can you see the chirp building up in amplitude as the two neutron stars approach merger? At approximately what time before merger does it become visible above the noise?
6. Compute the Q-transform for a 60-second window ending at the merger. Use `qrange=(8, 64)` and `frange=(20, 500)`. Describe what you see.

---

### Exercise 6 — Three-Detector Event: GW170814 ⭐⭐⭐

**Coding.** GW170814 was the first three-detector (H1, L1, V1) observation.

1. Look up the GPS time of GW170814 using `gwosc`.
2. Fetch 32 s of strain from H1, L1, and V1.
3. Whiten and band-pass all three.
4. Plot the whitened time series for all three detectors in separate panels (aligned in time), zoomed to ±0.3 s around the merger.
5. What is the expected time delay between detectors? (Hint: look up the inter-site light travel times for H1–L1, H1–V1, and L1–V1.)
6. Compute the Q-transform for each detector and compare the chirp tracks. Does Virgo show a clear signal? Why might the signal be weaker in Virgo?

---

### Exercise 7 — PSD Estimation: Method Comparison ⭐⭐⭐

**Analysis.** Compare different PSD estimation approaches.

1. Fetch 512 s of H1 data around GW150914.
2. Compute the PSD using Welch's method with:
   - `fftlength=1 s`, `overlap=0.5 s`
   - `fftlength=4 s`, `overlap=2 s`
   - `fftlength=16 s`, `overlap=8 s`
3. Plot all three ASDs on a single log-log plot in the range 10–2000 Hz.
4. Which has better frequency resolution? Which has less variance (smoother curve)? Explain the trade-off.
5. Use `method='median'` instead of `method='mean'` for the `fftlength=4 s` case. Is there a difference? Under what circumstances would median averaging be preferred?

---

### Exercise 8 — Spectral Line Identification ⭐⭐

**Analysis.** The detector ASD contains many spectral lines above the broadband noise floor.

1. Compute the ASD of H1 data during O1 (use the 512-second segment from Exercise 7).
2. Plot the ASD from 50 Hz to 500 Hz using a **linear** x-axis. (This makes it easier to see equally-spaced lines.)
3. Identify and tabulate all visible spectral lines between 50 and 300 Hz (at least 5). For each, record: frequency, approximate amplitude relative to the noise floor, and probable origin.
4. Apply a series of notch filters to suppress all identified lines. Plot the ASD before and after.
5. After notching: is the broadband noise floor changed? Does the notching affect the GW150914 signal in the band-passed time domain?

---

### Exercise 9 — GPS Time and Segment Analysis ⭐⭐

**Coding.** Analyse the data quality for a full 24-hour period.

1. Choose the GPS start of the day of GW150914: `t_start = 1126224000` (midnight UTC).
2. Query the H1_DATA Science-mode segments for this 24-hour period using `gwosc.timeline.get_segments()`.
3. Compute:
   - Total Science-mode time and duty cycle
   - Number of individual Science-mode segments
   - Duration of the longest and shortest segments
   - Mean and median segment duration
4. Plot a **segment timeline** showing all Science-mode segments as green horizontal bars (x-axis: hours from midnight).
5. Mark the GW150914 event time with a vertical red line. In which segment does it fall? How far from the segment start and end is the event?

---

### Exercise 10 — Full Pipeline: Your Own Event ⭐⭐⭐⭐

**Open-ended.** Choose any event from GWTC-2 or GWTC-3 that you find interesting.

1. Look up its GPS time and basic parameters (masses, distance, SNR) from the GWOSC event portal.
2. Fetch 32 s of strain from all available detectors.
3. Apply the full processing pipeline:
   - Resample to 2048 Hz (if originally 4096 Hz)
   - Whiten
   - Band-pass (choose the frequency range based on the expected chirp range for the source masses)
4. Plot the whitened, band-passed strain for each detector.
5. Compute and plot the Q-transform for each detector.
6. Write a short paragraph (3–5 sentences) about what makes this event scientifically interesting. Describe what you observe in the Q-transform.
7. *(Advanced)* Estimate the instantaneous frequency at merger using the chirp mass formula:
$$f_{\rm ISCO} \approx \frac{4400\,\text{Hz}}{M/M_\odot}$$
where $M$ is the total mass. Does the Q-transform track end near this frequency?

---

---
## 13. References

### Primary Software Documentation

1. **GWpy** — `gwpy`  
   Documentation: https://gwpy.github.io/docs/stable/  
   API reference: https://gwpy.github.io/docs/stable/api/  
   Source: https://github.com/gwpy/gwpy  
   Paper: Macas et al. (2022), *SoftwareX* 19, 101216. https://doi.org/10.1016/j.softx.2022.101216

2. **GWOSC Python package** — `gwosc`  
   Documentation: https://gwosc.readthedocs.io  
   Source: https://github.com/gwpy/gwosc

3. **SciPy Signal Processing** — `scipy.signal`  
   Filter design: https://docs.scipy.org/doc/scipy/reference/signal.html  
   Welch PSD: https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.welch.html

4. **NumPy FFT** — `numpy.fft`  
   Documentation: https://numpy.org/doc/stable/reference/routines.fft.html

### Gravitational-Wave Data Analysis

5. **GWOSC Tutorial: Signal Processing**  
   https://gwosc.org/tutorials/  
   The official GWOSC Jupyter tutorials on strain processing (highly recommended companion material).

6. **Open Data Workshop Materials**  
   https://gwosc.org/odw/  
   Annual workshops with lecture videos and guided notebooks on filtering, whitening, matched filtering.

7. **Gravitational-Wave Astronomy: An Introduction** (lecture notes)  
   Moore, Cole, Berry (2015). *Class. Quantum Grav.* 32, 015014.  
   https://doi.org/10.1088/0264-9381/32/1/015014

8. **LIGO Data and Noise** (review article)  
   Aasi et al. (2015). *Class. Quantum Grav.* 32, 074001.  
   https://doi.org/10.1088/0264-9381/32/7/074001

### Signal Processing Fundamentals

9. **Oppenheim & Schafer — Discrete-Time Signal Processing** (3rd ed.)  
   Pearson, 2009. The standard reference for digital signal processing.

10. **Proakis & Manolakis — Digital Signal Processing** (4th ed.)  
    Pearson, 2006. Accessible treatment of filtering, FFT, and spectral estimation.

11. **Welch (1967)** — The original paper on Welch's PSD method:  
    P. D. Welch, *IEEE Transactions on Audio and Electroacoustics* 15(2), 70–73.  
    https://doi.org/10.1109/TAU.1967.1161901

### GW Events and Catalogues

12. **GW150914 Detection Paper**  
    Abbott et al. (2016). *Physical Review Letters* 116, 061102.  
    https://doi.org/10.1103/PhysRevLett.116.061102

13. **GW170817 Detection Paper**  
    Abbott et al. (2017). *Physical Review Letters* 119, 161101.  
    https://doi.org/10.1103/PhysRevLett.119.161101

14. **GWTC-3 Catalogue**  
    Abbott et al. (2023). *Physical Review X* 13, 041039.  
    https://doi.org/10.1103/PhysRevX.13.041039

### Noise Characterisation

15. **Gravity Spy** — Machine-learning glitch classification  
    Website: https://www.gravityspy.org  
    Paper: Zevin et al. (2017), *Class. Quantum Grav.* 34, 064003.  
    https://doi.org/10.1088/1361-6382/aa5cea

16. **GW150914 Noise Analysis Companion Paper**  
    Abbott et al. (2016). *Class. Quantum Grav.* 33, 134001.  
    https://doi.org/10.1088/0264-9381/33/13/134001

### Q-Transform

17. **Brown & Puckette (1992)** — Original Q-transform paper:  
    J. C. Brown, *JASA* 92(5), 2698–2701.  
    https://doi.org/10.1121/1.404417

18. **Chatterji et al. (2004)** — Q-transform application to GW data:  
    S. Chatterji et al., *Class. Quantum Grav.* 21, S1809.  
    https://doi.org/10.1088/0264-9381/21/20/024

---

**Next lesson:** Lesson 4 — Matched Filtering and Signal-to-Noise Ratio